# 02 – EDA and Feature Engineering (Stage 4)
Find patterns with statistics and charts, then create and TEST engineered features.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from common import *

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
PAL = {"Fail": "#D1495B", "Withdrawn": "#8C5E58", "Pass": "#2E6FA3", "Distinction": "#3C9D6B"}
df = pd.read_csv(CLEAN_PATH, low_memory=False)
df["at_risk"] = df[TARGET].isin(AT_RISK).astype(int)
print(df.shape, "at-risk rate:", round(df.at_risk.mean() * 100, 2), "%")

## Figure 2 – target distribution

In [ ]:
cnt = df[TARGET].value_counts().reindex(CLASS_ORDER)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(cnt.index, cnt.values, color=[PAL[c] for c in cnt.index])
for i, v in enumerate(cnt.values):
    ax.text(i, v + 1500, f"{v:,}\n({v / cnt.sum() * 100:.1f}%)", ha="center", fontsize=9)
ax.set_ylabel("Number of students"); ax.set_title("Distribution of performance_status"); ax.set_ylim(0, cnt.max() * 1.18)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig02_target_distribution.png"); plt.close()

## Figure 3 – distributions of key numeric variables

In [ ]:
cols = ["attendance_rate", "midterm_score", "study_hours_week", "lms_logins_week", "prior_gpa", "assignment_submission_rate"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))
for ax, c in zip(axes.ravel(), cols):
    ax.hist(df[c].dropna(), bins=40, color="#2E6FA3", edgecolor="white")
    ax.axvline(df[c].median(), color="#D1495B", ls="--", lw=1.2, label=f"median {df[c].median():.1f}")
    ax.set_title(c); ax.legend(fontsize=8, frameon=False)
fig.suptitle("Distributions of key numeric variables (after cleaning)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig03_numeric_distributions.png"); plt.close()

## Figure 4 – engagement and performance by class (boxplots)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4.2))
for ax, c in zip(axes, ["attendance_rate", "midterm_score", "lms_logins_week", "study_hours_week"]):
    sns.boxplot(data=df, x=TARGET, y=c, order=CLASS_ORDER, palette=PAL, ax=ax, fliersize=1, hue=TARGET, legend=False)
    ax.set_xlabel(""); ax.set_title(c); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig04_boxplots_by_status.png"); plt.close()
by_class = df.groupby(TARGET)[["attendance_rate", "midterm_score", "assignment_avg_score", "quiz_avg_score",
                               "lms_logins_week", "study_hours_week", "prior_gpa", "late_submissions"]].mean().reindex(CLASS_ORDER).round(2)
by_class.to_csv(RES_DIR / "means_by_class.csv"); print(by_class.to_string())

## Figure 5 – at-risk rate by LMS activity

In [ ]:
df["lms_band"] = pd.cut(df["lms_logins_week"], [-1, 2, 5, 8, 11, 14, 40], labels=["0-2", "3-5", "6-8", "9-11", "12-14", "15+"])
rate = df.groupby("lms_band", observed=True)["at_risk"].mean() * 100
rate.to_csv(RES_DIR / "atrisk_by_lms.csv")
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(rate.index.astype(str), rate.values, color="#D1495B")
for i, v in enumerate(rate.values): ax.text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=9)
ax.set_xlabel("LMS logins per week"); ax.set_ylabel("At-risk rate (Fail + Withdrawn, %)")
ax.set_title("At-risk rate falls sharply as LMS activity rises"); ax.set_ylim(0, rate.max() * 1.15)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig05_atrisk_by_lms.png"); plt.close()

## Figure 6 – at-risk rate by program, connectivity and course difficulty

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, c in zip(axes, ["program", "internet_access", "course_difficulty"]):
    r = (df.groupby(c)["at_risk"].mean() * 100).sort_values()
    ax.barh(r.index, r.values, color="#2E6FA3")
    for i, v in enumerate(r.values): ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)
    ax.set_title(f"At-risk rate by {c}"); ax.set_xlabel("%"); ax.set_xlim(0, r.max() * 1.18)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig06_atrisk_by_category.png"); plt.close()
for c in ["program", "internet_access", "device_access", "course_difficulty", "enrollment_status",
          "scholarship_status", "financial_aid", "study_level", "semester"]:
    (df.groupby(c)["at_risk"].mean() * 100).round(2).to_csv(RES_DIR / f"atrisk_by_{c}.csv")

## Figure 7 – correlation heatmap

In [ ]:
num_cols = ["prior_gpa", "entry_exam_score", "attendance_rate", "study_hours_week", "lms_logins_week", "video_completion_rate",
            "assignment_submission_rate", "assignment_avg_score", "quiz_avg_score", "midterm_score", "late_submissions",
            "employment_hours_week", "commute_minutes", "support_sessions", "course_load_credits"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9.5, 7.5))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 6.5}, ax=ax, cbar_kws={"shrink": .8})
ax.set_title("Correlation between numeric variables")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig07_correlation_heatmap.png"); plt.close()
ordinal = df[TARGET].map({"Fail": 0, "Withdrawn": 0, "Pass": 1, "Distinction": 2})
tcorr = df[num_cols].corrwith(ordinal).sort_values(ascending=False).round(3)
tcorr.to_csv(RES_DIR / "corr_with_target.csv"); print(tcorr)

## Feature engineering
Nine row-wise features (nothing learned from other rows, so no leakage). See `common.add_features`.

In [ ]:
fe = add_features(df.drop(columns=["at_risk", "lms_band"]))
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, c in zip(axes, ["avg_assessment_score", "online_engagement", "attendance_x_submission"]):
    sns.boxplot(data=fe, x=TARGET, y=c, order=CLASS_ORDER, palette=PAL, ax=ax, fliersize=1, hue=TARGET, legend=False)
    ax.set_xlabel(""); ax.set_title(c); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig08_engineered_features.png"); plt.close()
fe_corr = fe[ENGINEERED].corrwith(ordinal).round(3).sort_values(); fe_corr.to_csv(RES_DIR / "engineered_corr_with_target.csv"); print(fe_corr)

## Do the engineered features add value? (ablation)
Same model, same 5-fold stratified CV on a 40,000-row sample: raw vs raw + engineered.

In [ ]:
sub, _ = train_test_split(fe, train_size=40000, stratify=fe[TARGET], random_state=RANDOM_STATE)
res = {}
for name, use_eng in [("raw features only", False), ("raw + engineered", True)]:
    feats, num, cat = split_columns(sub, use_eng)
    pre = ColumnTransformer([("num", SimpleImputer(strategy="median"), num),
                             ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                               ("o", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), cat)])
    pipe = Pipeline([("pre", pre), ("m", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, class_weight="balanced", random_state=RANDOM_STATE))])
    pred = cross_val_predict(pipe, sub[feats], sub[TARGET], cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
    res[name] = {"macro_f1": f1_score(sub[TARGET], pred, average="macro"),
                 "weighted_f1": f1_score(sub[TARGET], pred, average="weighted"), "n_features": len(feats)}
    print(name, res[name])
pd.DataFrame(res).T.to_csv(RES_DIR / "feature_ablation.csv")
json.dump({"engineered_features": ENGINEERED}, open(RES_DIR / "engineered_list.json", "w"))